In [1]:
!pip install arabic-reshaper python-bidi

from datasets import load_dataset,load_from_disk
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from transformers import TrainingArguments, Trainer
import torch
import torch.nn as nn
import torch.optim as optim
import transformers
from transformers import TrainingArguments, Trainer
import arabic_reshaper
from bidi.algorithm import get_display

In [2]:
!pip uninstall -y transformers accelerate peft trl
!pip cache purge

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Files removed: 12


In [3]:
!pip install transformers==4.41.2
!pip install accelerate==0.30.1
!pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 106.6 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 15.6 MB/s eta 0:00:00


In [2]:
def CvtTxt_4_CmdPrint(txt):
    reshaped_text = arabic_reshaper.reshape(txt)
    bidi_text = get_display(reshaped_text)
    return bidi_text

In [3]:
!pip install -q gdown
import gdown

file_id = "1Rf-z2JPLormEDE-WpzDfXjjoYRz2mC5I"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "Bert_QA.zip", quiet=True)
!unzip "/content/Bert_QA.zip" > /dev/null 2>&1
# !mv -f "/content/anomaly detection final" "/content/datasets" > /dev/null 2>&1
!rm "/content/Bert_QA.zip" > /dev/null 2>&1

In [4]:
model_checkpoint = r"./Bert_QA/models/checkpoint-30000"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
QA_dataset = load_from_disk('./Bert_QA/DataSet/persian_qa')
print(QA_dataset.keys())
train_size = len(QA_dataset['train'])
validation_size = len(QA_dataset['validation'])

print("Train set size:", train_size)
print("Validation set size:", validation_size)

dict_keys(['train', 'validation'])
Train set size: 9008
Validation set size: 930


In [5]:
# Hyperparameters
max_length = 512 # The maximum length of a feature (question and context)
doc_stride = 256 # The authorized overlap between two part of the context when splitting it is needed.
batch_size = 8
lr = 3e-5
epoch = 3

In [6]:
def prepare_train_features(examples):
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,)

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")
    print("...........................tokenized_examples.pop(overflow_to_sample_mapping).................................")
    print(len(sample_mapping) , type (sample_mapping),len(offset_mapping),offset_mapping)

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []
    for i, offsets in enumerate(offset_mapping):
        # We will label impossible answers with the index of the CLS token.
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        # Grab the sequence corresponding to that example (to know what is the context and what is the question).
        sequence_ids = tokenized_examples.sequence_ids(i)
        # One example can give several spans, this is the index of the example containing this span of text.
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        # If no answers are given, set the cls_index as answer.
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # Start/end character index of the answer in the text.
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])
            # Start token index of the current span in the text.
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            # End token index of the current span in the text.
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1
            # Detect if the answer is out of the span (in which case this feature is labeled with the CLS index).
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                # Otherwise move the token_start_index and token_end_index to the two ends of the answer.
                # Note: we could go after the last offset if the answer is the last word (edge case).
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)


                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

In [7]:
print("--- Number of trainable parameters ================================")
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters in the model: {params}")

--- Number of trainable parameters ================================
Number of trainable parameters in the model: 162252290


In [8]:
print("--- Start==========================================================")
print(QA_dataset)
print(CvtTxt_4_CmdPrint(str(QA_dataset['train'][10])))

--- Start==========================================================
DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 9008
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 930
    })
})
{'id': 11, 'title': 'ﻪﻧﺎﯾﺍﺭ ﻡﻮﻠﻋ', 'context': 'ﯽﻣ ﻪﺘﻔﮔ ﯽﺗﺎﻌﻟﺎﻄﻣ ﻪﻋﻮﻤﺠﻣ ﻪﺑ ﺮﺗﻮﯿﭙﻣﺎﮐ ﻡﻮﻠﻋ ﺎﯾ ﻪﻧﺎﯾﺍﺭ ﻡﻮﻠﻋ\u200cﺵﻭﺭ ،ﯼﺮﻈﻧ ﯼﺎﻫﺎﻨﺑﺮﯾﺯ ﻪﺑ ﻪﮐ ﺩﻮﺷ\u200cﯽﻣ ﻪﻧﺎﯾﺍﺭ ﺯﺍ ﻩﺩﺎﻔﺘﺳﺍ ﯽﮕﻧﻮﮕﭼ ﻭ ﺖﺧﺎﺳ ﻭ ﯽﺣﺍﺮﻃ ﯼﺎﻫ\u200cﯽﻣ ﺍﺭ ﺮﺗﻮﯿﭙﻣﺎﮐ ﻡﻮﻠﻋ ﻪﺘﺷﺭ.ﺪﻧﺯﺍﺩﺮﭘ\u200cﻪﺘﺷﺭﺮﯾﺯ ﻪﺑ ﻥﺍﻮﺗ\u200cﻪﺘﺷﺭﺮﯾﺯ ﻦﯾﺍ ﺯﺍ ﯽﻀﻌﺑ .ﺩﺮﮐ ﻢﯿﺴﻘﺗ ﯼﺭﺎﯿﺴﺑ ﯽﻠﻤﻋ ﻭ ﯼﺮﻈﻧ ﯼﺎﻫ\u200cﻥﺁ ﻥﺩﻮﺑ ﻞﺣ ﻞﺑﺎﻗ ﻭ ﯽﺗﺎﺒﺳﺎﺤﻣ ﺕﻼﮑﺸﻣ ﯽﺳﺎﺳﺍ ﺹﺍﻮﺧ ﻪﮐ) ﯽﺗﺎﺒﺳﺎﺤﻣ ﯽﮔﺪﯿﭽﯿﭘ ﻪﯾﺮﻈﻧ ﺮﯿﻈﻧ ،ﺎﻫ\u200cﯽﻣ ﯽﺳﺭﺮﺑ ﺍﺭ ﺎﻫ\u200cﻪﺘﺷﺭﺮﯾﺯ ﻪﮐ ﺖﺳﺍ ﯽﻟﺎﺣ ﺭﺩ ﻦﯾﺍ ،ﺪﻨﺘﺴﻫ ﯽﻋﺍﺰﺘﻧﺍ ﺭﺎﯿﺴﺑ (ﺪﻨﮐ\u200cﻪﺘﺷﺭﺮﯾﺯ ﺮﺜﮐﺍ .ﺪﻧﺭﺍﺩ ﺪﯿﮐﺄﺗ ﯽﻌﻗﺍﻭ ﯼﺎﯿﻧﺩ ﺭﺩ ﺮﺗ ﺲﻤﻟ ﻞﺑﺎﻗ ﯼﺎﻫﺩﺮﺑﺭﺎﮐ ﯽﺳﺭﺮﺑ ﻪﺑ ﯼﺮﺗﻮﯿﭙﻣﺎﮐ ﮏﯿﻓﺍﺮﮔ ﺪﻨﻧﺎﻣ ﺮﮕﯾﺩ ﯼﺎﻫ\u200cﺶﻟﺎﭼ ﺮﺑ ﺮﺗﻮﯿﭙﻣﺎﮐ ﻡﻮﻠﻋ ﯼﺎﻫ\u200cﻩﺩﺍﺩ ﺕﺭﺎﺒﻋ ﻪﮐ ﯽﻤﻠﻋ ﻪﺴﺳﺆﻣ ﻦﯿﻟﻭﺍ  .ﺪﻧﺭﺍﺩ ﺰﮐﺮﻤﺗ ﺕﺎﺒﺳﺎﺤﻣ ﯼﺍ

In [9]:
print("--- End  ==========================================================")
tokenized_ds = QA_dataset.map(prepare_train_features, batched=True, remove_columns=QA_dataset["train"].column_names)

--- End  ==========================================================


In [10]:
print("2--- Start==========================================================")
print(tokenized_ds)
print(CvtTxt_4_CmdPrint(str(tokenized_ds['train'][10])))

2--- Start==========================================================
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 9008
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 930
    })
})
{'input_ids': [2, 4568, 4458, 6075, 4442, 2787, 1350, 4, 4458, 8285, 2880, 4458, 6075, 2789, 3855, 13352, 3538, 2886, 2800, 2789, 41749, 6412, 1348, 5124, 3841, 1379, 3043, 1379, 7346, 2988, 2791, 8285, 9590, 1012, 4336, 4458, 6075, 2803, 2944, 2789, 64399, 2794, 6412, 1379, 3850, 3398, 4629, 2830, 1012, 4322, 2791, 2802, 64399, 2809, 1348, 5163, 5395, 9678, 12518, 1006, 2800, 6594, 4664, 3966, 12518, 1379, 3496, 3803, 3798, 2950, 2803, 3640, 2980, 1007, 3177, 16134, 3060, 1348, 2802, 2786, 3747, 2806, 2800, 64399, 2794, 2972, 3331, 8085, 10305, 2789, 3640, 8946, 3496, 11093, 3088, 2786, 520

In [11]:
print("2--- End  ==========================================================")
print(model)

2--- End  ==========================================================
BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(100000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=76

In [12]:
import transformers

In [13]:
print(transformers.__version__)
print(transformers.__file__)

4.41.2
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [14]:
from transformers import TrainingArguments
print(TrainingArguments)

<class 'transformers.training_args.TrainingArguments'>


In [15]:
args = TrainingArguments(
    f"result",
    evaluation_strategy = "epoch",
    logging_strategy="steps",  # Log after each training step
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epoch,
    weight_decay=0.0001,
    report_to="none",
    save_total_limit=2  # Save the last 2 checkpoints)
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [16]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    tokenizer=tokenizer)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.095600,6.628837
2,0.040200,6.476626
3,0.007000,6.822759


TrainOutput(global_step=3378, training_loss=0.04268165182403871, metrics={'train_runtime': 1675.837, 'train_samples_per_second': 16.126, 'train_steps_per_second': 2.016, 'total_flos': 3854804900318784.0, 'train_loss': 0.04268165182403871, 'epoch': 3.0})

In [18]:
print(model)

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(100000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
trainer.save_model("./models/QA_Bert-1403-03-25")

In [29]:
from transformers import BertTokenizerFast, BertForQuestionAnswering

tokenizer_1 = BertTokenizerFast.from_pretrained("models/QA_Bert-1403-03-25")
model_1 = BertForQuestionAnswering.from_pretrained("models/QA_Bert-1403-03-25")

In [30]:
from transformers import pipeline

In [37]:
qa_pipeline = pipeline("question-answering", model=model, tokenizer="HooshvareLab/bert-fa-base-uncased", device="cuda")

text = r"""سلام  دوره علم داده با پایتون در کارخانه نوآوری مشهد برگزار میگردد با تدریس دکتر علی ملایی در روز های یکشنبه و سه شنبه در ساعت های شانزده الی بیست """
questions = ["مکان برگزاری دوره کجاست؟", "دکتر علی ملایی کیست؟", "کارخانه نوآوری کجاست؟"]

result = qa_pipeline({
  'question': "پایتخت ایران کجاست؟",
  'context': "تهران پایتخت است."
})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [35]:
for question in questions:
    print(qa_pipeline({"context": text, "question": question}))

{'score': 0.7998173832893372, 'start': 33, 'end': 52, 'answer': 'کارخانه نوآوری مشهد'}
{'score': 1.5780825200740166e-15, 'start': 0, 'end': 66, 'answer': 'سلام  دوره علم داده با پایتون در کارخانه نوآوری مشهد برگزار میگردد'}
{'score': 1.0, 'start': 33, 'end': 52, 'answer': 'کارخانه نوآوری مشهد'}


In [38]:
print("پاسخ:", result['answer'])

پاسخ: تهران
